# Use a bulk dataset as a reference

Bulk RNA-seq of sorted populations can train viewmastR. `splat_bulk_reference()`
simulates single cells from each bulk profile ("splatting"): each cell gets a library size
drawn from the query's cell sizes, then molecules drawn from the bulk profile. This
tutorial follows the R article
[How to use a bulk dataset as a reference](https://furlan-lab.github.io/viewmastR/articles/BulkReference.html).

| Role | Dataset |
|---|---|
| Bulk reference | sorted haematopoietic populations and AML cells (Corces et al. 2016, GEO GSE74246), as shipped with viewmastR |
| Query 1 | healthy bone marrow atlas (Granja et al. 2019, "Greenleaf reference") |
| Query 2 | a large query of your own (h5ad, local file) |

In [ ]:
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import viewmastr as vm
import tutorial_data as td

sc.set_figure_params(dpi=80, frameon=False)
label = td.config().get("greenleaf_label", "celltype")
greenleaf = td.greenleaf_reference()

## Build the bulk object

Columns of the count table are `<donor>-<population>`. As in the R article, the first
reference drops the leukaemic blasts, leukaemia stem cells (LSC) and the rHSC samples.

In [ ]:
counts = td.corces_bulk()
meta = pd.DataFrame([c.split("-", 1) for c in counts.columns], columns=["donor", "celltype"], index=counts.columns)
bulk = ad.AnnData(counts.T.to_numpy(dtype=np.float32), obs=meta, var=pd.DataFrame(index=counts.index))
healthy = bulk[~bulk.obs.celltype.str.match("Blast|rHSC|LSC")].copy()
print(healthy.obs.celltype.value_counts().to_string())

## Splat 200 cells per bulk sample

Library sizes come from the Greenleaf cells (`dist="sc-direct"`). The simulated object is
a normal single-cell dataset: normalise, embed and cluster it as usual.

In [ ]:
sim = vm.splat_bulk_reference(greenleaf, healthy, N=200)
s = sim.copy()
sc.pp.normalize_total(s, target_sum=1e4)
sc.pp.log1p(s)
sc.pp.highly_variable_genes(s, n_top_genes=1000)
sc.pp.pca(s, n_comps=50, mask_var="highly_variable")
sc.pp.neighbors(s, n_pcs=20)
sc.tl.umap(s, min_dist=0.6)
vm.palette.set_colors(s, "celltype")
sc.pl.umap(s, color=["celltype", "donor"], ncols=2)

## Classify single cells with the bulk reference

Train on the simulated cells and label the Greenleaf atlas. The atlas has its own labels,
so the confusion matrix shows how the bulk populations map onto single-cell types.

In [ ]:
def shared_variable_genes(query, ref):
    for a in (query, ref):
        vm.calculate_feature_dispersion(a)
        vm.select_features(a, top_n=10_000, logmean_ul=-1, logmean_ll=-8)
    vgr = set(vm.get_selected_features(ref))
    return [g for g in vm.get_selected_features(query) if g in vgr]

vg = shared_variable_genes(greenleaf, sim)
vm.viewmastR(greenleaf, sim, "celltype", selected_features=vg)
for key in ("viewmastR_pred", label):
    vm.palette.set_colors(greenleaf, key)
sc.pl.umap(greenleaf, color=["viewmastR_pred", label], ncols=2, wspace=0.6)
vm.confusion_matrix(greenleaf.obs["viewmastR_pred"], greenleaf.obs[label],
                    cols=vm.palette.colors([*greenleaf.obs["viewmastR_pred"], *greenleaf.obs[label]]))
plt.show()

## A leukaemia-aware bulk reference

The R article classifies a patient sample with a reference built from leukaemia bulk
profiles. Here the bulk reference keeps every population of the Corces data, including
the blasts and LSC, and 10 cells are splatted per sample. The query is a random sample
of 20,000 cells of a large query of your own (set `large_query` in
`data_paths.json`; any raw-count h5ad works).

In [ ]:
cfg = td.config()
big = td.large_query(backed=True)
rows = np.sort(np.random.default_rng(0).choice(big.n_obs, 20_000, replace=False))
aml = ad.AnnData(big.X[rows], var=big.var.copy())
gene_col = cfg.get("large_query_gene_column")
if gene_col:
    aml.var_names = aml.var[gene_col].astype(str).to_numpy()
    aml.var_names_make_unique(join=".")
aml.obs_names = [f"cell{i}" for i in range(aml.n_obs)]  # no original cell barcodes
sim_all = vm.splat_bulk_reference(aml, bulk, N=10)
vg = shared_variable_genes(aml, sim_all)
_ = vm.viewmastR(aml, sim_all, "celltype", selected_features=vg)

Embed the query cells and show the predictions. As in R, labels with fewer than 100 cells
are grouped as "Other".

In [ ]:
sc.pp.normalize_total(aml, target_sum=1e4)
sc.pp.log1p(aml)
sc.pp.highly_variable_genes(aml, n_top_genes=2000)
sc.pp.pca(aml, n_comps=30, mask_var="highly_variable")
sc.pp.neighbors(aml)
sc.tl.umap(aml)
n = aml.obs["viewmastR_pred"].value_counts()
aml.obs["pred_thresh"] = aml.obs["viewmastR_pred"].astype(str).where(
    aml.obs["viewmastR_pred"].astype(str).isin(n[n >= 100].index), "Other")
vm.palette.set_colors(aml, "pred_thresh")
sc.pl.umap(aml, color="pred_thresh")
print(aml.obs["pred_thresh"].value_counts().to_string())

Not translated: the R article's donor/recipient chimerism panel needs per-cell genotype
calls, which these datasets do not carry.

## Versions

In [ ]:
from importlib.metadata import version
for pkg in ("viewmastr", "scanpy", "anndata", "numpy", "scipy"):
    print(f"{pkg:10s} {version(pkg)}")